# Avant le TP1 : travailler avec les notebooks du lab

Le TP1 se fait dans un notebook. Aucun script utilitaire n'est caché derrière : chaque cellule est soit une **commande de terminal**, soit du **Python avec `confluent-kafka`**. Cette démonstration montre les deux sur un même sujet, la liste des topics :

1. une commande Kafka lancée dans le conteneur `tools` ;
2. une commande `docker compose` lancée dans le dossier `lab/` ;
3. la même information obtenue avec le client Python ;
4. le terminal intégré de JupyterLab, équivalent à ces cellules.

Toutes les démonstrations suivent ce principe : chaque étape montre la version **terminal**, puis la version **Python**.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

## 1. Une commande du conteneur `tools`

Une ligne qui commence par `!` est envoyée au terminal. `docker exec kafka-tools` lance la commande dans le conteneur d'administration, qui contient les outils `kafka-*.sh`. C'est exactement ce que vous feriez dans un terminal ouvert sur le poste.

`BS` est la liste d'adresses des brokers sur le listener INTERNAL. Elle est déjà définie dans l'environnement du notebook ; on la récupère en Python, avec la configuration du client, pour les cellules suivantes. Dans une ligne `!`, `$BS` est remplacé par la variable Python du même nom.

In [1]:
import os

from confluent_kafka import KafkaException
from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}
print("Bootstrap INTERNAL :", BS)

Bootstrap INTERNAL : kafka-1:19092,kafka-2:19092,kafka-3:19092


**En terminal**

In [2]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --list

__consumer_offsets
loadgen


## 2. Une commande `docker compose`

Les commandes `docker compose` se lancent dans le dossier `lab/`, qui contient `docker-compose.yml`. `FORMATION_DIR` est le dossier de la formation, défini dans l'environnement du notebook. Avec `%%bash`, toute la cellule est un petit script de terminal : on peut y mettre un `cd`, un tube, plusieurs lignes. Arrêter un nœud, lire ses traces ou consulter un fichier de configuration se font de cette façon. Le client Python n'a pas d'équivalent : l'état des conteneurs se lit en terminal.

In [3]:
%%bash
cd $FORMATION_DIR/lab
docker compose ps --format '{{.Name}} {{.Health}}' | grep -E '^kafka-[0-9]'

kafka-1 healthy
kafka-2 healthy
kafka-3 healthy


## 3. La même information en Python

`AdminClient` est le client d'administration de la bibliothèque `confluent-kafka`. On lui donne la configuration `CONF` ; `list_topics()` renvoie un objet Python que l'on parcourt, filtre et compte. Le client est gardé dans une variable : il doit vivre jusqu'à la réponse du cluster. Si le cluster ne répond pas dans le délai, `list_topics()` lève une `KafkaException`, que l'on attrape pour afficher son code.

**En Python**

In [4]:
admin = AdminClient(CONF)
try:
    md = admin.list_topics(timeout=10)
except KafkaException as e:
    print("Cluster injoignable :", e.args[0].name())
    raise

for name, topic in sorted(md.topics.items()):
    print("%-20s %2d partition(s)" % (name, len(topic.partitions)))

__consumer_offsets   50 partition(s)
loadgen               6 partition(s)


Un exemple d'exploitation : combien de partitions compte le cluster ? En terminal, on compte les lignes `Partition:` de la description de tous les topics.

**En terminal**

In [5]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe | grep -c "Partition: "

56


**En Python** : `md.topics` est un dictionnaire `{name: description}`, et chaque description porte un dictionnaire `partitions`.

In [6]:
total = sum(len(topic.partitions) for topic in md.topics.values())
print(total, "partitions")

56 partitions


## 4. Le terminal de JupyterLab

Les cellules `!` et `%%bash` ne sont qu'un confort : le même résultat s'obtient dans un terminal. Dans JupyterLab, **File > New > Terminal** ouvre un terminal dans le conteneur `jupyter`, avec les mêmes commandes :

```
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --list
cd $FORMATION_DIR/lab && docker compose ps
```

Utilisez la forme que vous préférez, cellule ou terminal.

## La méthode du TP1

Ouvrez `tps/jour1/tp1_cluster/enonce.ipynb`. Pour chaque étape :

1. lisez l'explication ;
2. exécutez l'exemple en ligne de commande et lisez sa sortie ;
3. complétez la zone `# TODO` en Python ;
4. exécutez la cellule de vérification.

Les preuves collectées alimentent le procès-verbal de recette affiché en fin de notebook.